# User Story 2 — Multi-Prompt LLM-as-a-Judge (EN→Mandarin)

Compares three judge prompts on Qwen3-8B (4-bit):

- **P0_baseline** — original Sprint 51 prompt (unchanged)
- **P1_anchored_fewshot** — anchored rubric with hard score caps + 4 human-scored calibration examples (consistency technique)
- **P2_explain_rate** — fixed 3-step analysis written *before* the score (validity technique)

Phase A: validity vs gold set + challenge-set failure detection at temperature 0.
Phase B: repeated runs at temperature 0.2 for consistency.

**Setup:** Runtime → Change runtime type → T4 GPU. Set `CONFIG["dry_run"] = True` in Cell 3 for a 5-minute smoke test first.
Full run ≈ 2 h on a free T4.

**Note for the report:** the 50-sample gold set is generated in Cell 5 and is author-labeled, not independently human-annotated.

In [1]:
# Cell 1 — Install (Colab, GPU runtime)
!pip install -q unsloth
!nvidia-smi --query-gpu=name,memory.total --format=csv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.4/82.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 108.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 121.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 23.2 MB/s eta 0:00:00
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Cell 2 — Imports
import os, re, gc, json, math, random, statistics, warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import cohen_kappa_score
import transformers
from transformers import set_seed
from unsloth import FastLanguageModel

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1635: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# Cell 3 — Config
CONFIG = {
    "experiment_name": "us2_multiprompt_judge",
    "seed": 53,
    "data": {
        "gold_set_csv_path": "/content/data/gold_set_en_cmn.csv",  # generated in Cell 5
        "direction": "EN_CMN",
        "max_gold_samples": 50,
    },
    "judge": {
        "model_name": "unsloth/Qwen3-8B-unsloth-bnb-4bit",  # "unsloth/Qwen3-4B-unsloth-bnb-4bit" for ~2x speed
        "max_seq_length": 4096,
        "max_new_tokens": 400,
        "load_in_4bit": True,
        "phase_a_temperature": 0.0,
        "phase_a_runs": 1,
        "phase_b_temperature": 0.2,
        "phase_b_runs": 3,
        "base_generation_seed": 5300,
    },
    "consistency": {"high_disagreement_range": 2.0, "high_disagreement_std": 1.0},
    "output": {"root": "/content/outputs"},
    "dry_run": False,   # True = 5 gold + 5 challenge samples
    "dry_run_n": 5,
}

PROMPT_IDS = ["P0_baseline", "P1_anchored_fewshot", "P2_explain_rate"]

OUT = Path(CONFIG["output"]["root"])
for f in [OUT, OUT/"raw_runs", OUT/"aggregated", OUT/"summaries", OUT/"challenge", OUT/"tables"]:
    f.mkdir(parents=True, exist_ok=True)
Path("/content/data").mkdir(exist_ok=True)

random.seed(CONFIG["seed"]); np.random.seed(CONFIG["seed"]); torch.manual_seed(CONFIG["seed"])
print("Output:", OUT)

Output: /content/outputs


In [4]:
# Cell 4 — Helpers
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)

def safe_float(x):
    try:
        return None if pd.isna(x) else float(x)
    except Exception:
        return None

In [5]:
# Cell 5 — Generate the gold set (50 EN->CMN samples, author-labeled)
GOLD_ROWS = [
 # --- near-perfect ---
 ("CMN_001","The library closes at eight on weekdays.","图书馆工作日晚上八点关门。",9.8,"correct"),
 ("CMN_002","Please keep your receipt for returns.","请保留收据以便退货。",9.7,"correct"),
 ("CMN_003","The new bridge will open to traffic next spring.","新桥将于明年春天通车。",9.8,"correct"),
 ("CMN_004","She studied chemistry at a university in Shanghai.","她在上海的一所大学学习化学。",9.6,"correct"),
 ("CMN_005","Water boils at 100 degrees Celsius at sea level.","在海平面，水在100摄氏度沸腾。",9.5,"correct"),
 ("CMN_006","The meeting was moved from Tuesday to Thursday.","会议从周二改到了周四。",9.7,"correct"),
 ("CMN_007","Our flight was delayed by two hours because of fog.","由于大雾，我们的航班延误了两个小时。",9.8,"correct"),
 ("CMN_008","The museum offers free admission on the first Sunday of every month.","博物馆每月第一个周日免费开放。",9.6,"correct"),
 ("CMN_009","He forgot his umbrella on the train.","他把雨伞忘在火车上了。",9.7,"correct"),
 ("CMN_010","This medicine should be taken twice a day after meals.","这种药应每天饭后服用两次。",9.6,"correct"),
 ("CMN_011","The company hired forty new engineers last year.","该公司去年招聘了四十名新工程师。",9.8,"correct"),
 ("CMN_012","Turn off the lights when you leave the room.","离开房间时请关灯。",9.7,"correct"),
 ("CMN_013","The river flooded several villages after a week of heavy rain.","连续一周的暴雨过后，河水淹没了几个村庄。",9.5,"correct"),
 ("CMN_014","Tickets can be purchased online or at the box office.","门票可以在网上或售票处购买。",9.6,"correct"),
 ("CMN_015","The children planted trees along the school fence.","孩子们沿着学校的围栏种了树。",9.5,"correct"),
 ("CMN_016","The economy grew by three percent in the second quarter.","第二季度经济增长了百分之三。",9.6,"correct"),
 ("CMN_017","The password must contain at least eight characters.","密码必须包含至少八个字符。",9.3,"correct"),
 ("CMN_018","The printer on the third floor is out of order.","三楼的打印机坏了。",9.0,"correct_colloquial"),
 ("CMN_019","He apologized for arriving late to the interview.","他为面试迟到而道歉。",9.2,"correct"),
 ("CMN_020","The bakery sells bread, cakes, and fresh pastries.","这家面包店卖面包、蛋糕和新鲜的糕点。",9.4,"correct"),
 # --- minor issues ---
 ("CMN_021","I'll let you know as soon as I hear back from them.","一旦我收到他们的回复，我就会让你知道。",8.5,"slightly literal"),
 ("CMN_022","The soup needs a little more salt.","这个汤需要一点点更多的盐。",7.0,"awkward phrasing"),
 ("CMN_023","The report is due by the end of the month.","报告在月底之前到期。",7.5,"'due' rendered as 到期"),
 ("CMN_024","The hotel is a ten-minute walk from the station.","酒店离车站走路十分钟。",8.5,"colloquial but fine"),
 ("CMN_025","Please fill in the form in block capitals.","请用大写字母填写表格。",8.8,"minor nuance loss"),
 ("CMN_026","We ran out of milk, so I went to the store.","我们牛奶用完了，所以我去了商店。",8.3,"slightly literal"),
 ("CMN_027","The old factory was converted into apartments.","旧工厂被转换成公寓。",7.5,"转换 slightly unnatural"),
 ("CMN_028","It's not that I don't like him; I just don't trust him.","不是我不喜欢他，我只是不相信他。",8.5,"相信 vs 信任 nuance"),
 # --- noticeable issues ---
 ("CMN_029","She bought three tickets for Saturday's concert.","她买了周六音乐会的票。",6.0,"omitted 'three'"),
 ("CMN_030","The software update will be released on March 3.","软件更新将于3月份发布。",5.5,"day omitted"),
 ("CMN_031","The restaurant was crowded, but the service was quick.","餐厅很拥挤，服务也很快。",6.5,"contrast lost"),
 ("CMN_032","He asked whether the train stops at the airport.","他问火车是否停在机场。",6.5,"ambiguous 停在"),
 ("CMN_033","Please do not feed the animals.","请不要动物喂食。",5.5,"ungrammatical"),
 ("CMN_034","The lecture covered the causes of the First World War.","讲座涵盖了世界大战的原因。",5.5,"omitted 'First'"),
 ("CMN_035","The patient should avoid heavy lifting for six weeks.","病人应该六周避免举重。",6.0,"举重 = the sport"),
 # --- major / critical ---
 ("CMN_036","Most of the students passed the exam.","所有学生都通过了考试。",4.5,"most -> all"),
 ("CMN_037","Add two cups of flour and one egg.","加入两杯面粉。",4.5,"omitted egg"),
 ("CMN_038","The train leaves at 7:45 in the morning.","火车早上8:45出发。",3.5,"wrong time"),
 ("CMN_039","The package was delivered to Mr. Chen.","包裹送到了王先生那里。",3.5,"wrong entity"),
 ("CMN_040","She was born in 1992 in Guangzhou.","她1929年出生于广州。",3.5,"wrong year"),
 ("CMN_041","Please restart your router and try again.","请 restart 您的 router 然后再试一次。",3.5,"English leakage"),
 ("CMN_042","The weather will be cold tomorrow.","明天天氣會很冷。",4.0,"Traditional script"),
 ("CMN_043","Where did you put the keys?","你將鑰匙放喺邊度？",3.0,"Cantonese"),
 ("CMN_044","Do not open the door while the machine is running.","机器运行时请打开门。",2.5,"negation dropped"),
 ("CMN_045","Sales fell by 12% last quarter.","上个季度销售额增长了12%。",2.5,"direction reversed"),
 ("CMN_046","Break a leg tonight!","今晚摔断一条腿！",2.5,"idiom literal"),
 ("CMN_047","He sold his car and bought a motorcycle.","他买了一辆车，卖了一辆摩托车。",2.5,"verbs swapped"),
 ("CMN_048","Good morning, everyone.","Dajia zaoshang hao.",2.0,"Pinyin"),
 ("CMN_049","The conference will be held in Chengdu next year.","The conference will be held in Chengdu next year.",1.5,"untranslated"),
 ("CMN_050","The medicine must not be given to children under six.","这种药必须给六岁以下的儿童服用。",1.5,"negation flipped"),
]
gold_df = pd.DataFrame(GOLD_ROWS, columns=["sample_id","source_en","mt_output","overall_gold","note"])
gold_df.insert(1, "direction", "EN_CMN")
gold_df.to_csv(CONFIG["data"]["gold_set_csv_path"], index=False)
print(len(gold_df), "gold rows written. Score distribution:")
print(pd.cut(gold_df["overall_gold"], [0,4.5,6.9,8.9,10]).value_counts().sort_index())

@dataclass
class GoldSample:
    sample_id: str; direction: str; source_en: str; mt_output: str; overall_gold: float
    reference_zh: Optional[str] = None

def load_gold_set(path):
    df = pd.read_csv(path)
    out = []
    for _, r in df.iterrows():
        g = safe_float(r["overall_gold"])
        if g is None: continue
        ref = None if "reference_zh" not in df.columns or pd.isna(r.get("reference_zh")) else str(r["reference_zh"])
        out.append(GoldSample(str(r["sample_id"]), str(r.get("direction","EN_CMN")),
                              str(r["source_en"]), str(r["mt_output"]), g, ref))
    return out

def validate_gold_set(samples):
    issues, seen = [], set()
    for s in samples:
        if s.sample_id in seen: issues.append(f"dup {s.sample_id}")
        seen.add(s.sample_id)
        if not 1 <= s.overall_gold <= 10: issues.append(f"{s.sample_id} out of range")
        if not s.source_en.strip() or not s.mt_output.strip(): issues.append(f"{s.sample_id} empty")
    return {"n_total": len(samples), "issues": issues}

50 gold rows written. Score distribution:
overall_gold
(0.0, 4.5]     15
(4.5, 6.9]      7
(6.9, 8.9]      8
(8.9, 10.0]    20
Name: count, dtype: int64


In [6]:
# Cell 6 — Aggregation + agreement utilities (unchanged from Sprint 51)
SCORE_MIN, SCORE_MAX = 1.0, 10.0
ZERO_OUTLIER_STD_THRESHOLD = 2.0
HIGH_DISAGREEMENT_STD = 1.5
MIN_VALID_RUNS_FOR_SCORE = 2
MALFORMED_RUN_FAILURE_FRACTION = 0.5
GEMBA_LAMBDA = 0.5

@dataclass
class RunResult:
    score: Optional[float]; parse_ok: bool; raw_response: Optional[str] = None

def aggregate_runs(runs):
    valid, malformed = [], 0
    for r in runs:
        if not r.parse_ok or r.score is None or not (SCORE_MIN <= r.score <= SCORE_MAX): malformed += 1
        else: valid.append(float(r.score))
    n = len(runs); mf = malformed / n if n else 1.0
    fail = {"final_score":None,"confidence":0.0,"flag":"judge_failed","aggregation_method":None,
            "n_total_runs":n,"n_valid_runs":len(valid),"malformed_fraction":round(mf,3),"valid_scores":valid}
    if n == 1:
        if len(valid) == 1:
            return {"final_score":round(valid[0],3),"confidence":1.0,"flag":"ok","aggregation_method":"single_run",
                    "n_total_runs":1,"n_valid_runs":1,"malformed_fraction":0.0,"valid_scores":valid}
        return fail
    if mf >= MALFORMED_RUN_FAILURE_FRACTION or len(valid) < MIN_VALID_RUNS_FOR_SCORE:
        return fail
    med = statistics.median(valid); std = statistics.pstdev(valid)
    clean = [s for s in valid if not (s <= 1.0 and std > 0 and (med - s) > ZERO_OUTLIER_STD_THRESHOLD*std)]
    outliers = [s for s in valid if s not in clean]
    use = clean or valid
    med2 = statistics.median(use); std2 = statistics.pstdev(use) if len(use) > 1 else 0.0
    high = std2 >= HIGH_DISAGREEMENT_STD
    if high:
        final, method = med2, "median_fallback_high_disagreement"
    else:
        w = [1/(1 + GEMBA_LAMBDA*abs(s-med2)) for s in use]
        final, method = sum(a*b for a,b in zip(w,use))/sum(w), "gemba_weighted_mean"
    flag = "ok"
    if outliers: flag = "zero_outliers_excluded"
    if high: flag = "high_disagreement" if flag == "ok" else flag + "+high_disagreement"
    return {"final_score":round(final,3),"confidence":round(max(0.0,1-std2/5.0),3),"flag":flag,
            "aggregation_method":method,"n_total_runs":n,"n_valid_runs":len(valid),
            "malformed_fraction":round(mf,3),"valid_scores":valid}

def compute_agreement(gold, judged):
    g, j = np.array(gold, float), np.array(judged, float)
    if len(g) < 2:
        return {"pearson_r":np.nan,"spearman_rho":np.nan,"mae":np.nan,"weighted_kappa":np.nan,"n":len(g)}
    gi = np.clip(np.round(g),1,10).astype(int); ji = np.clip(np.round(j),1,10).astype(int)
    return {"pearson_r":round(float(pearsonr(g,j)[0]),3),
            "spearman_rho":round(float(spearmanr(g,j)[0]),3),
            "mae":round(float(np.mean(np.abs(g-j))),3),
            "weighted_kappa":round(float(cohen_kappa_score(gi,ji,weights="quadratic")),3),
            "n":len(g)}

In [7]:
# Cell 7 — Prompts: P0 baseline (unchanged), P1 anchored few-shot, P2 explain-then-rate
COMMON_OUTPUT_SCHEMA = '''
{
  "score": 1-10,
  "feedback": "brief explanation",
  "compliance": {
    "fully_translated": true,
    "target_language_mandarin": true,
    "simplified_chinese": true,
    "english_leakage": false,
    "pinyin_or_romanization": false,
    "cantonese_or_non_mandarin": false
  },
  "errors": [
    {"category": "mistranslation|omission|addition|entity|number|negation|terminology|fluency|grammar|language|script|source_leakage|format|other",
     "severity": "minor|major|critical",
     "description": "short description"}
  ]
}
'''.strip()

# "analysis" is placed first so the score is derived from written checks, not intuition.
EXPLAIN_RATE_SCHEMA = '''
{
  "analysis": "STEP1 compliance: ... STEP2 meaning: ... STEP3 fluency: ...",
  "score": 1-10,
  "feedback": "one-sentence summary",
  "compliance": {
    "fully_translated": true,
    "target_language_mandarin": true,
    "simplified_chinese": true,
    "english_leakage": false,
    "pinyin_or_romanization": false,
    "cantonese_or_non_mandarin": false
  },
  "errors": [
    {"category": "mistranslation|omission|addition|entity|number|negation|terminology|fluency|grammar|language|script|source_leakage|format|other",
     "severity": "minor|major|critical",
     "description": "short description"}
  ]
}
'''.strip()

RUBRIC = '''
SCORING RUBRIC — choose the band first, then the exact number:
10 = Perfect: complete meaning, natural Simplified Mandarin, nothing to fix.
9  = Near-perfect: one negligible stylistic choice; a professional accepts as-is.
8  = Good: 1-2 minor wording or fluency issues; meaning fully intact.
7  = Acceptable: several minor issues or awkward phrasing; meaning intact.
6  = Usable with edits: one noticeable error that does not change the core message.
5  = Flawed: a meaningful omission, addition, or ungrammatical phrase affecting part of the message.
4  = Poor: one major factual error (wrong number/entity/time/date, important omission) OR Traditional/mixed script.
3  = Bad: core meaning wrong or reversed (negation flipped, opposite outcome), even if fluent.
2  = Unusable: mostly untranslated, dominant English leakage, Pinyin, or Cantonese instead of Mandarin.
1  = No translation value.
HARD CAPS (never exceed): untranslated / Pinyin / wrong language <= 2;
negation or meaning reversal <= 3; wrong number/entity/date, Traditional or mixed script,
major English leakage <= 4. Valid paraphrases that keep the meaning are NOT errors.
'''.strip()

# Anchor examples: none of these sentences appear in the gold or challenge sets.
FEWSHOT_EXAMPLES = '''
CALIBRATION EXAMPLES (human-scored — match this scale):

Example A
English: The museum opens at nine and closes at five.
Mandarin: 博物馆九点开门，五点关门。
Output: {"score": 10, "feedback": "Complete, accurate, natural.", "compliance": {"fully_translated": true, "target_language_mandarin": true, "simplified_chinese": true, "english_leakage": false, "pinyin_or_romanization": false, "cantonese_or_non_mandarin": false}, "errors": []}

Example B
English: She has lived in Berlin for three years.
Mandarin: 她在柏林住了三年了，到现在还住在那里。
Output: {"score": 7, "feedback": "Meaning preserved; second clause is an unnecessary addition.", "compliance": {"fully_translated": true, "target_language_mandarin": true, "simplified_chinese": true, "english_leakage": false, "pinyin_or_romanization": false, "cantonese_or_non_mandarin": false}, "errors": [{"category": "addition", "severity": "minor", "description": "added clause not in source"}]}

Example C
English: Turn left at the second traffic light.
Mandarin: 在第三个红绿灯右转。
Output: {"score": 3, "feedback": "Fluent but ordinal and direction are both wrong; instruction is unusable.", "compliance": {"fully_translated": true, "target_language_mandarin": true, "simplified_chinese": true, "english_leakage": false, "pinyin_or_romanization": false, "cantonese_or_non_mandarin": false}, "errors": [{"category": "number", "severity": "major", "description": "second -> third"}, {"category": "mistranslation", "severity": "critical", "description": "left -> right"}]}

Example D
English: Your order has been shipped.
Mandarin: Your order 已经 shipped 了。
Output: {"score": 2, "feedback": "Mostly untranslated with English leakage.", "compliance": {"fully_translated": false, "target_language_mandarin": true, "simplified_chinese": true, "english_leakage": true, "pinyin_or_romanization": false, "cantonese_or_non_mandarin": false}, "errors": [{"category": "source_leakage", "severity": "critical", "description": "key words left in English"}]}
'''.strip()

def wrap_qwen(system_text, user_text):
    return ("<|im_start|>system\n" + system_text.strip() +
            "\n<|im_end|>\n<|im_start|>user\n" + user_text.strip() +
            "\n<|im_end|>\n<|im_start|>assistant\n/no_think\n")

def _user_block(english, mandarin):
    return f"English source:\n{english}\n\nCandidate Mandarin:\n{mandarin}"

def build_p0_baseline(english, mandarin):
    system = '''
You are a strict professional translation evaluator.
Evaluate the Mandarin translation for accuracy, fluency, and fidelity.
Return ONLY valid JSON in this exact format:
{"score": <number from 1 to 10>, "feedback": "<short explanation>"}
Do not add any extra text.
Do not think aloud.
'''
    return wrap_qwen(system, f"English source:\n{english}\n\nMandarin translation:\n{mandarin}")

def build_p1_anchored_fewshot(english, mandarin):
    system = f'''
You are a strict professional English-to-Mandarin translation evaluator.
Score the candidate using ONLY the rubric below, and keep your scores consistent
with the calibration examples: candidates of similar quality must get similar scores.

{RUBRIC}

{FEWSHOT_EXAMPLES}

Return ONLY valid JSON using this schema:
{COMMON_OUTPUT_SCHEMA}

Do not add text outside JSON. Do not think aloud.
'''
    return wrap_qwen(system, _user_block(english, mandarin))

def build_p2_explain_rate(english, mandarin):
    system = f'''
You are a strict professional English-to-Mandarin translation evaluator.

Fill in the "analysis" field FIRST, in this fixed order, then derive the score from it:
STEP 1 compliance — fully translated? Mandarin? Simplified Chinese? any Pinyin, English words, or Cantonese?
STEP 2 meaning — compare every number, date, name, quantity, negation and clause with the source;
       name any mistranslation, omission, or addition.
STEP 3 fluency — grammar and naturalness, considered only after Steps 1-2.

The score MUST follow from the analysis:
9-10 Steps 1-3 all clean | 7-8 only minor Step 3 issues | 5-6 one noticeable Step 2 issue, core message intact |
3-4 major Step 2 error or Step 1 script failure | 1-2 untranslated, Pinyin, wrong language, or meaning reversed.
A fluent sentence with wrong meaning must score low.

Return ONLY valid JSON using this schema (keep "analysis" under 60 words):
{EXPLAIN_RATE_SCHEMA}

Do not add text outside JSON.
'''
    return wrap_qwen(system, _user_block(english, mandarin))

PROMPT_BUILDERS = {
    "P0_baseline": build_p0_baseline,
    "P1_anchored_fewshot": build_p1_anchored_fewshot,
    "P2_explain_rate": build_p2_explain_rate,
}
print(list(PROMPT_BUILDERS))

['P0_baseline', 'P1_anchored_fewshot', 'P2_explain_rate']


In [8]:
# Cell 8 — Challenge set
CHALLENGE_ROWS = [
{"sample_id":"CH01_correct","source_en":"Please close the window before you leave.","candidate_zh":"请在离开前关上窗户。","category":"correct","critical_failure":False,"expected_max_score":10},
{"sample_id":"CH02_valid_paraphrase","source_en":"The meeting has been postponed until Friday.","candidate_zh":"会议改到周五举行。","category":"valid_paraphrase","critical_failure":False,"expected_max_score":10},
{"sample_id":"CH03_minor_fluency","source_en":"I will call you when I arrive.","candidate_zh":"当我到达的时候，我会给你打电话。","category":"minor_fluency","critical_failure":False,"expected_max_score":8},
{"sample_id":"CH04_major_mistranslation","source_en":"The train leaves at six in the morning.","candidate_zh":"火车晚上六点出发。","category":"mistranslation","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH05_omission","source_en":"She bought apples, oranges, and two bottles of water.","candidate_zh":"她买了苹果和橙子。","category":"omission","critical_failure":True,"expected_max_score":5},
{"sample_id":"CH06_addition","source_en":"He opened the door.","candidate_zh":"他生气地打开门，然后离开了房间。","category":"addition","critical_failure":True,"expected_max_score":5},
{"sample_id":"CH07_wrong_entity","source_en":"Maria sent the report to David.","candidate_zh":"玛丽亚把报告发给了彼得。","category":"entity","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH08_wrong_number","source_en":"The package contains 15 tablets.","candidate_zh":"这个包装里有五片药片。","category":"number","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH09_negation","source_en":"Do not restart the computer.","candidate_zh":"请重新启动电脑。","category":"negation","critical_failure":True,"expected_max_score":3},
{"sample_id":"CH10_untranslated_7_francs","source_en":"7 francs.","candidate_zh":"7 francs.","category":"untranslated","critical_failure":True,"expected_max_score":2},
{"sample_id":"CH11_untranslated_sentence","source_en":"The doctor will see you tomorrow.","candidate_zh":"The doctor will see you tomorrow.","category":"untranslated","critical_failure":True,"expected_max_score":2},
{"sample_id":"CH12_english_leakage","source_en":"Please upload the final report today.","candidate_zh":"请今天 upload 最终报告。","category":"source_leakage","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH13_traditional","source_en":"The weather is very good today.","candidate_zh":"今天天氣非常好。","category":"traditional_script","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH14_mixed_script","source_en":"Please confirm the information.","candidate_zh":"请確認这些信息。","category":"mixed_script","critical_failure":True,"expected_max_score":5},
{"sample_id":"CH15_cantonese","source_en":"What are you doing?","candidate_zh":"你做緊乜嘢？","category":"cantonese","critical_failure":True,"expected_max_score":3},
{"sample_id":"CH16_pinyin","source_en":"Thank you very much.","candidate_zh":"Feichang ganxie ni.","category":"pinyin","critical_failure":True,"expected_max_score":2},
{"sample_id":"CH17_fluent_but_wrong","source_en":"The company reported a loss this quarter.","candidate_zh":"该公司本季度实现了盈利。","category":"fluent_but_wrong","critical_failure":True,"expected_max_score":3},
{"sample_id":"CH18_wrong_date","source_en":"The deadline is October 12.","candidate_zh":"截止日期是10月21日。","category":"number","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH19_partial_translation","source_en":"Click Save, then restart the application.","candidate_zh":"点击“保存”，then restart the application.","category":"source_leakage","critical_failure":True,"expected_max_score":4},
{"sample_id":"CH20_idiom","source_en":"Let's call it a day.","candidate_zh":"让我们称它为一天。","category":"idiom_mistranslation","critical_failure":True,"expected_max_score":4},
]
challenge_df = pd.DataFrame(CHALLENGE_ROWS)
challenge_df.to_csv(OUT/"challenge"/"challenge_set.csv", index=False)
print(len(challenge_df), "challenge rows")

20 challenge rows


In [9]:
# Cell 9 — Judge model
class MultiPromptJudgeModel:
    def __init__(self, model_name, max_seq_length=4096, max_new_tokens=400, load_in_4bit=True):
        self.max_seq_length, self.max_new_tokens = max_seq_length, max_new_tokens
        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_name, max_seq_length=max_seq_length, dtype=None, load_in_4bit=load_in_4bit)
        FastLanguageModel.for_inference(self.model)
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

    def clean_output(self, text):
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
        text = re.sub(r"</?think>", "", text).strip()
        if text.startswith("```"):
            text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
            text = re.sub(r"\s*```$", "", text)
        return text.strip()

    def _extract_json(self, text):
        try:
            o = json.loads(text); return o if isinstance(o, dict) else None
        except Exception: pass
        s, e = text.find("{"), text.rfind("}")
        if s >= 0 and e > s:
            try:
                o = json.loads(text[s:e+1]); return o if isinstance(o, dict) else None
            except Exception: pass
        return None

    def parse_output(self, raw):
        cleaned = self.clean_output(raw)
        parsed = self._extract_json(cleaned)
        score, feedback, compliance, errors, analysis = None, cleaned, {}, [], None
        strict = parsed is not None
        if parsed is not None:
            try: score = float(parsed.get("score"))
            except Exception: score = None
            feedback = parsed.get("feedback", cleaned)
            compliance = parsed.get("compliance") or {}
            errors = parsed.get("errors") or []
            analysis = parsed.get("analysis")
        if score is None:
            m = re.search(r'(?<!\d)(10(?:\.0+)?|[1-9](?:\.\d+)?)(?!\d)', cleaned)
            if m and SCORE_MIN <= float(m.group(1)) <= SCORE_MAX: score = float(m.group(1))
        ok = score is not None and SCORE_MIN <= score <= SCORE_MAX
        return {"score": round(score,2) if ok else None, "feedback": feedback, "analysis": analysis,
                "strict_json_ok": strict, "parse_ok": bool(ok),
                "compliance": compliance if isinstance(compliance, dict) else {},
                "errors": errors if isinstance(errors, list) else []}

    def evaluate(self, english, mandarin, prompt_id, temperature=0.0, seed=None):
        if seed is not None: set_seed(seed)
        prompt = PROMPT_BUILDERS[prompt_id](english, mandarin)
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True,
                                    max_length=self.max_seq_length).to(self.model.device)
            kw = dict(**inputs, max_new_tokens=self.max_new_tokens, do_sample=temperature > 0,
                      use_cache=True, pad_token_id=self.tokenizer.pad_token_id,
                      eos_token_id=self.tokenizer.eos_token_id)
            if temperature > 0: kw["temperature"] = temperature
            with torch.no_grad():
                out = self.model.generate(**kw)
            raw = self.tokenizer.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
            p = self.parse_output(raw)
            return {**p, "error": None if p["parse_ok"] else "no valid 1-10 score", "raw_output": raw}
        except Exception as e:
            return {"score":None,"feedback":None,"analysis":None,"strict_json_ok":False,"parse_ok":False,
                    "compliance":{},"errors":[],"error":str(e),"raw_output":None}

In [10]:
# Cell 10 — Load data + model
gold_samples = load_gold_set(CONFIG["data"]["gold_set_csv_path"])
gold_samples = [s for s in gold_samples if s.direction == CONFIG["data"]["direction"]][:CONFIG["data"]["max_gold_samples"]]
print("Validation:", validate_gold_set(gold_samples))

gold_examples = [{"sample_id":s.sample_id,"source_en":s.source_en,"candidate_zh":s.mt_output,
                  "gold_score":s.overall_gold} for s in gold_samples]
challenge_examples = [dict(r) for r in CHALLENGE_ROWS]
if CONFIG["dry_run"]:
    gold_examples = gold_examples[:CONFIG["dry_run_n"]]
    challenge_examples = challenge_examples[:CONFIG["dry_run_n"]]
print("Gold:", len(gold_examples), "Challenge:", len(challenge_examples))

judge = MultiPromptJudgeModel(CONFIG["judge"]["model_name"], CONFIG["judge"]["max_seq_length"],
                              CONFIG["judge"]["max_new_tokens"], CONFIG["judge"]["load_in_4bit"])
print("Loaded:", CONFIG["judge"]["model_name"])

Validation: {'n_total': 50, 'issues': []}
Gold: 50 Challenge: 20
==((====))==  Unsloth 2026.10.3: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loaded: unsloth/Qwen3-8B-unsloth-bnb-4bit


In [11]:
# Cell 11 — Runner + aggregation
def flatten_errors(errors):
    cats, sevs = [], []
    for e in errors or []:
        if isinstance(e, dict):
            c = str(e.get("category","")).strip().lower(); s = str(e.get("severity","")).strip().lower()
            if c: cats.append(c)
            if s: sevs.append(s)
    return cats, sevs

COMPLIANCE_KEYS = ["fully_translated","target_language_mandarin","simplified_chinese",
                   "english_leakage","pinyin_or_romanization","cantonese_or_non_mandarin"]

def run_examples(examples, prompt_ids, n_runs, temperature, phase, dataset_name):
    rows, total, done = [], len(examples)*len(prompt_ids)*n_runs, 0
    for ei, ex in enumerate(examples):
        for pi, pid in enumerate(prompt_ids):
            for ri in range(n_runs):
                seed = CONFIG["judge"]["base_generation_seed"] + ei*1000 + pi*100 + ri
                r = judge.evaluate(ex["source_en"], ex["candidate_zh"], pid, temperature, seed)
                cats, sevs = flatten_errors(r.get("errors"))
                row = {"dataset":dataset_name,"phase":phase,"sample_id":ex["sample_id"],"prompt_id":pid,
                       "run_id":ri+1,"temperature":temperature,"seed":seed,
                       "source_en":ex["source_en"],"candidate_zh":ex["candidate_zh"],
                       "gold_score":ex.get("gold_score"),"judge_score":r.get("score"),
                       "parse_ok":r.get("parse_ok",False),"strict_json_ok":r.get("strict_json_ok",False),
                       "feedback":r.get("feedback"),"analysis":r.get("analysis"),"raw_output":r.get("raw_output"),
                       "error_message":r.get("error"),
                       "error_categories":json.dumps(cats,ensure_ascii=False),
                       "error_severities":json.dumps(sevs,ensure_ascii=False),
                       "n_errors":len(cats),"has_major_error":"major" in sevs,"has_critical_error":"critical" in sevs,
                       **{k:(r.get("compliance") or {}).get(k) for k in COMPLIANCE_KEYS}}
                for k in ["category","critical_failure","expected_max_score"]:
                    if k in ex: row[k] = ex[k]
                rows.append(row); done += 1
                if done % 25 == 0 or done == total: print(f"{phase} {dataset_name}: {done}/{total}")
    return pd.DataFrame(rows)

def aggregate_run_dataframe(raw):
    out = []
    for (ds, ph, sid, pid), g in raw.groupby(["dataset","phase","sample_id","prompt_id"], dropna=False):
        agg = aggregate_runs([RunResult(safe_float(x["judge_score"]), bool(x["parse_ok"])) for _, x in g.iterrows()])
        valid = [float(x) for x in g.loc[g["parse_ok"]==True,"judge_score"].dropna()]
        first = g.iloc[0]
        row = {"dataset":ds,"phase":ph,"sample_id":sid,"prompt_id":pid,
               "source_en":first["source_en"],"candidate_zh":first["candidate_zh"],"gold_score":first.get("gold_score"),
               "final_score":agg["final_score"],"judge_confidence":agg["confidence"],"judge_flag":agg["flag"],
               "aggregation_method":agg["aggregation_method"],"n_total_runs":agg["n_total_runs"],
               "n_valid_runs":agg["n_valid_runs"],"valid_scores":json.dumps(agg["valid_scores"]),
               "run_mean":np.mean(valid) if valid else np.nan,
               "run_std":statistics.pstdev(valid) if len(valid)>1 else (0.0 if valid else np.nan),
               "run_range":(max(valid)-min(valid)) if valid else np.nan,
               "exact_agreement":bool(valid) and len(set(round(v,6) for v in valid))==1,
               "strict_json_success_rate":float(g["strict_json_ok"].mean()),
               "parse_success_rate":float(g["parse_ok"].mean())}
        for c in ["category","critical_failure","expected_max_score"]:
            if c in g.columns: row[c] = first.get(c)
        out.append(row)
    return pd.DataFrame(out)

In [12]:
# Cell 12 — Phase A: gold validity (temperature 0)
phase_a_gold_raw = run_examples(gold_examples, PROMPT_IDS, CONFIG["judge"]["phase_a_runs"],
                                CONFIG["judge"]["phase_a_temperature"], "A", "gold")
phase_a_gold_raw.to_csv(OUT/"raw_runs"/"phase_a_gold_raw.csv", index=False)
phase_a_gold_agg = aggregate_run_dataframe(phase_a_gold_raw)
phase_a_gold_agg.to_csv(OUT/"aggregated"/"phase_a_gold_aggregated.csv", index=False)

def gold_prompt_metrics(agg):
    rows = []
    for pid, g in agg.groupby("prompt_id"):
        u = g[g["final_score"].notna() & g["gold_score"].notna() & (g["judge_flag"]!="judge_failed")]
        rows.append({"prompt_id":pid,"n_total":len(g),"n_usable":len(u),
                     "coverage":round(len(u)/len(g),3) if len(g) else 0.0,
                     "strict_json_success":round(float(g["strict_json_success_rate"].mean()),3),
                     "parse_success":round(float(g["parse_success_rate"].mean()),3),
                     **compute_agreement(u["gold_score"].astype(float), u["final_score"].astype(float))})
    return pd.DataFrame(rows).sort_values(["spearman_rho","mae"], ascending=[False,True])

phase_a_gold_metrics = gold_prompt_metrics(phase_a_gold_agg)
phase_a_gold_metrics.to_csv(OUT/"tables"/"phase_a_gold_validity_metrics.csv", index=False)
display(phase_a_gold_metrics)

A gold: 25/150
A gold: 50/150
A gold: 75/150
A gold: 100/150
A gold: 125/150
A gold: 150/150


,prompt_id,n_total,n_usable,coverage,strict_json_success,parse_success,pearson_r,spearman_rho,mae,weighted_kappa,n
1,P1_anchored_fewshot,50,50,1.0,0.04,1.0,0.757,0.725,1.53,0.693,50
0,P0_baseline,50,50,1.0,0.02,1.0,0.699,0.696,1.91,0.605,50
2,P2_explain_rate,50,50,1.0,0.92,1.0,0.665,0.678,1.87,0.619,50


In [13]:
# Cell 13 — Phase A: challenge set
phase_a_challenge_raw = run_examples(challenge_examples, PROMPT_IDS, CONFIG["judge"]["phase_a_runs"],
                                     CONFIG["judge"]["phase_a_temperature"], "A", "challenge")
phase_a_challenge_raw.to_csv(OUT/"raw_runs"/"phase_a_challenge_raw.csv", index=False)
phase_a_challenge_agg = aggregate_run_dataframe(phase_a_challenge_raw)
phase_a_challenge_agg.to_csv(OUT/"aggregated"/"phase_a_challenge_aggregated.csv", index=False)

FAILURE_CATEGORY_ALIASES = {
    "mistranslation":{"mistranslation","semantic","accuracy"},"idiom_mistranslation":{"mistranslation","semantic","accuracy"},
    "fluent_but_wrong":{"mistranslation","semantic","accuracy"},"omission":{"omission"},"addition":{"addition"},
    "entity":{"entity","mistranslation"},"number":{"number","mistranslation"},"negation":{"negation","mistranslation"},
    "untranslated":{"language","source_leakage","format"},"source_leakage":{"source_leakage","language"},
    "traditional_script":{"script","language"},"mixed_script":{"script","language"},
    "cantonese":{"language","terminology"},"pinyin":{"language","source_leakage","format"},
}
def parse_list_cell(x):
    try:
        v = json.loads(x); return [str(a).lower() for a in v] if isinstance(v, list) else []
    except Exception: return []

def challenge_metrics(raw, agg):
    rows = []
    for pid in sorted(agg["prompt_id"].unique()):
        a = agg[agg["prompt_id"]==pid]; r = raw[raw["prompt_id"]==pid]
        crit, nonc = a[a["critical_failure"]==True], a[a["critical_failure"]==False]
        recall = float((crit["final_score"].notna() & (crit["final_score"] <= crit["expected_max_score"])).mean()) if len(crit) else np.nan
        fpr = float((nonc["final_score"] <= 6).mean()) if len(nonc) else np.nan
        hits = []
        for _, x in r[r["critical_failure"]==True].iterrows():
            cat = str(x["category"]).lower(); cats = set(parse_list_cell(x["error_categories"]))
            flag = {"untranslated": x.get("fully_translated") is False or x.get("target_language_mandarin") is False,
                    "source_leakage": x.get("english_leakage") is True,
                    "traditional_script": x.get("simplified_chinese") is False,
                    "mixed_script": x.get("simplified_chinese") is False,
                    "pinyin": x.get("pinyin_or_romanization") is True,
                    "cantonese": x.get("cantonese_or_non_mandarin") is True}.get(cat, False)
            hits.append(bool(cats & FAILURE_CATEGORY_ALIASES.get(cat, set())) or bool(flag))
        rows.append({"prompt_id":pid,"n_challenge":len(a),"n_critical":len(crit),
                     "critical_failure_score_recall":round(recall,3),
                     "structured_failure_recall":round(float(np.mean(hits)),3) if hits else np.nan,
                     "false_positive_rate":round(fpr,3),
                     "mean_critical_score":round(float(crit["final_score"].mean()),3),
                     "mean_noncritical_score":round(float(nonc["final_score"].mean()),3),
                     "strict_json_success":round(float(r["strict_json_ok"].mean()),3),
                     "parse_success":round(float(r["parse_ok"].mean()),3)})
    return pd.DataFrame(rows).sort_values(["critical_failure_score_recall","structured_failure_recall"], ascending=False)

phase_a_challenge_metrics = challenge_metrics(phase_a_challenge_raw, phase_a_challenge_agg)
phase_a_challenge_metrics.to_csv(OUT/"tables"/"phase_a_challenge_metrics.csv", index=False)
display(phase_a_challenge_metrics)
display(phase_a_challenge_agg.pivot(index="sample_id", columns="prompt_id", values="final_score")
        .join(challenge_df.set_index("sample_id")[["category","expected_max_score"]]))

A challenge: 25/60
A challenge: 50/60
A challenge: 60/60


,prompt_id,n_challenge,n_critical,critical_failure_score_recall,structured_failure_recall,false_positive_rate,mean_critical_score,mean_noncritical_score,strict_json_success,parse_success
2,P2_explain_rate,20,17,0.588,0.647,0.0,3.647,10.0,0.8,1.0
1,P1_anchored_fewshot,20,17,0.412,0.000,0.0,4.824,10.0,0.0,1.0
0,P0_baseline,20,17,0.235,0.000,0.0,6.000,10.0,0.0,1.0


,P0_baseline,P1_anchored_fewshot,P2_explain_rate,category,expected_max_score
sample_id,,,,,
CH01_correct,10.0,10.0,10.0,correct,10
CH02_valid_paraphrase,10.0,10.0,10.0,valid_paraphrase,10
CH03_minor_fluency,10.0,10.0,10.0,minor_fluency,8
CH04_major_mistranslation,5.0,8.0,6.0,mistranslation,4
CH05_omission,5.0,6.0,6.0,omission,5
CH06_addition,5.0,6.0,5.0,addition,5
CH07_wrong_entity,5.0,6.0,1.0,entity,4
CH08_wrong_number,5.0,5.0,5.0,number,4
CH09_negation,1.0,3.0,1.0,negation,3


In [14]:
# Cell 14 — Phase B: repeated runs at temperature 0.2 (consistency)
phase_b_gold_raw = run_examples(gold_examples, PROMPT_IDS, CONFIG["judge"]["phase_b_runs"],
                                CONFIG["judge"]["phase_b_temperature"], "B", "gold")
phase_b_gold_raw.to_csv(OUT/"raw_runs"/"phase_b_gold_raw.csv", index=False)
phase_b_gold_agg = aggregate_run_dataframe(phase_b_gold_raw)
phase_b_gold_agg.to_csv(OUT/"aggregated"/"phase_b_gold_aggregated.csv", index=False)

def consistency_metrics(agg):
    rows = []
    for pid, g in agg.groupby("prompt_id"):
        u = g[g["n_valid_runs"]>0]
        high = (u["run_range"] >= CONFIG["consistency"]["high_disagreement_range"]) | \
               (u["run_std"] >= CONFIG["consistency"]["high_disagreement_std"])
        rows.append({"prompt_id":pid,"n_samples":len(g),
                     "mean_run_std":round(float(u["run_std"].mean()),3),
                     "mean_run_range":round(float(u["run_range"].mean()),3),
                     "exact_agreement_rate":round(float(u["exact_agreement"].mean()),3),
                     "high_disagreement_rate":round(float(high.mean()),3),
                     "mean_judge_confidence":round(float(u["judge_confidence"].mean()),3),
                     "mean_parse_success":round(float(u["parse_success_rate"].mean()),3),
                     "mean_strict_json_success":round(float(u["strict_json_success_rate"].mean()),3)})
    return pd.DataFrame(rows).sort_values(["mean_run_std","mean_run_range"])

phase_b_consistency = consistency_metrics(phase_b_gold_agg)
phase_b_consistency.to_csv(OUT/"tables"/"phase_b_consistency_metrics.csv", index=False)
phase_b_gold_metrics = gold_prompt_metrics(phase_b_gold_agg)
phase_b_gold_metrics.to_csv(OUT/"tables"/"phase_b_gold_validity_metrics.csv", index=False)
display(phase_b_consistency); display(phase_b_gold_metrics)

B gold: 25/450
B gold: 50/450
B gold: 75/450
B gold: 100/450
B gold: 125/450
B gold: 150/450
B gold: 175/450
B gold: 200/450
B gold: 225/450
B gold: 250/450
B gold: 275/450
B gold: 300/450
B gold: 325/450
B gold: 350/450
B gold: 375/450
B gold: 400/450
B gold: 425/450
B gold: 450/450


,prompt_id,n_samples,mean_run_std,mean_run_range,exact_agreement_rate,high_disagreement_rate,mean_judge_confidence,mean_parse_success,mean_strict_json_success
1,P1_anchored_fewshot,50,0.019,0.04,0.98,0.02,0.996,1.0,0.053
0,P0_baseline,50,0.085,0.18,0.90,0.04,0.983,1.0,0.033
2,P2_explain_rate,50,0.585,1.24,0.78,0.14,0.934,1.0,0.867


,prompt_id,n_total,n_usable,coverage,strict_json_success,parse_success,pearson_r,spearman_rho,mae,weighted_kappa,n
1,P1_anchored_fewshot,50,50,1.0,0.053,1.0,0.756,0.715,1.522,0.693,50
0,P0_baseline,50,50,1.0,0.033,1.0,0.720,0.700,1.875,0.599,50
2,P2_explain_rate,50,50,1.0,0.867,1.0,0.523,0.524,2.202,0.503,50


In [15]:
# Cell 15 — Final comparison, summary, download
def minmax(s, higher_is_better=True):
    s = pd.to_numeric(s, errors="coerce")
    if s.notna().sum() == 0: return pd.Series([0.0]*len(s), index=s.index)
    lo, hi = s.min(), s.max()
    out = pd.Series([1.0]*len(s), index=s.index) if math.isclose(float(lo), float(hi)) else (s-lo)/(hi-lo)
    return out if higher_is_better else 1-out

final_table = (phase_a_gold_metrics[["prompt_id","spearman_rho","pearson_r","mae","weighted_kappa","strict_json_success"]]
    .rename(columns={"spearman_rho":"A_spearman","pearson_r":"A_pearson","mae":"A_mae","weighted_kappa":"A_kappa","strict_json_success":"A_json_success"})
    .merge(phase_a_challenge_metrics[["prompt_id","critical_failure_score_recall","structured_failure_recall","false_positive_rate"]], on="prompt_id")
    .merge(phase_b_consistency[["prompt_id","mean_run_std","mean_run_range","exact_agreement_rate","high_disagreement_rate"]], on="prompt_id")
    .merge(phase_b_gold_metrics[["prompt_id","spearman_rho","mae","weighted_kappa"]]
           .rename(columns={"spearman_rho":"B_spearman","mae":"B_mae","weighted_kappa":"B_kappa"}), on="prompt_id"))

final_table["final_rank_score"] = (
    0.20*minmax(final_table["A_spearman"]) + 0.15*minmax(final_table["A_kappa"]) + 0.10*minmax(final_table["A_mae"], False)
  + 0.20*minmax(final_table["critical_failure_score_recall"]) + 0.10*minmax(final_table["structured_failure_recall"])
  + 0.10*minmax(final_table["mean_run_std"], False) + 0.05*minmax(final_table["mean_run_range"], False)
  + 0.05*minmax(final_table["high_disagreement_rate"], False) + 0.05*minmax(final_table["A_json_success"]))
final_table = final_table.sort_values("final_rank_score", ascending=False).reset_index(drop=True)
final_table.to_csv(OUT/"tables"/"final_prompt_comparison.csv", index=False)
display(final_table)

(phase_a_challenge_agg.groupby(["prompt_id","category"], as_index=False)
 .agg(n=("sample_id","count"), mean_score=("final_score","mean"), expected_max=("expected_max_score","mean"))
 .to_csv(OUT/"tables"/"challenge_by_category.csv", index=False))

save_json({"experiment":CONFIG["experiment_name"],"judge_model":CONFIG["judge"]["model_name"],
           "n_gold":len(gold_examples),"n_challenge":len(challenge_examples),"prompt_ids":PROMPT_IDS,
           "phase_a_gold":phase_a_gold_metrics.to_dict("records"),
           "phase_a_challenge":phase_a_challenge_metrics.to_dict("records"),
           "phase_b_consistency":phase_b_consistency.to_dict("records"),
           "phase_b_gold":phase_b_gold_metrics.to_dict("records"),
           "final_comparison":final_table.to_dict("records")},
          OUT/"summaries"/"experiment_summary.json")

!cd /content && zip -r -q outputs.zip outputs data
from google.colab import files
files.download("/content/outputs.zip")

,prompt_id,A_spearman,A_pearson,A_mae,A_kappa,A_json_success,critical_failure_score_recall,structured_failure_recall,false_positive_rate,mean_run_std,mean_run_range,exact_agreement_rate,high_disagreement_rate,B_spearman,B_mae,B_kappa,final_rank_score
0,P1_anchored_fewshot,0.725,0.757,1.53,0.693,0.04,0.412,0.000,0.0,0.019,0.04,0.98,0.02,0.715,1.522,0.693,0.751394
1,P2_explain_rate,0.678,0.665,1.87,0.619,0.92,0.588,0.647,0.0,0.585,1.24,0.78,0.14,0.524,2.202,0.503,0.384390
2,P0_baseline,0.696,0.699,1.91,0.605,0.02,0.235,0.000,0.0,0.085,0.18,0.90,0.04,0.700,1.875,0.599,0.250768


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>